# Lab 2 — Resistencia a la compresión del hormigón

**Objetivo:** entrenar un modelo de regresión que prediga la resistencia a compresión (MPa) de una mezcla de hormigón a partir de su dosificación, y usar la importancia de variables para razonar sobre qué ingredientes dominan el resultado.

**Recorrido de este notebook:**
1. Contexto del problema y del dataset
2. Carga del dataset
3. Calidad de datos
4. Estadísticas descriptivas
5. Distribución del target (Resistencia)
6. Correlación entre variables
7. Relación física Agua vs Resistencia
8. Partición train / test
9. Random Forest (hiperparámetros)
10. Feature importance y validación visual


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

%matplotlib inline
sns.set_theme(style='whitegrid')
print('✅ Entorno listo (labs/.venv).')


## Contexto del dataset (UCI)

| Variable | Unidad | En obra significa… |
|----------|--------|-------------------|
| Cemento, Escoria, CenizaVolante | kg/m³ | Dosificación de ligantes y adiciones |
| Agua | kg/m³ | Relación agua/cemento (W/C) |
| Superplastificante | kg/m³ | Reducir agua sin perder trabajabilidad |
| AgregadoGrueso / Fino | kg/m³ | Grava y arena |
| Edad | días | Curado antes del ensayo |
| **Resistencia** | **MPa** | **Variable a predecir (y)** |

Fuente: Prof. I-Cheng Yeh · 1 030 mezclas · regresión supervisada.

Detalle ampliado: [`data/DATOS.md`](data/DATOS.md).


## 1. Contexto del hormigón y Machine Learning

Predecir la resistencia en MPa es un problema de **regresión**: la salida es un número continuo, no una etiqueta. Trabajar en MPa (y no con una etiqueta "bueno/malo") es lo que permite comparar directamente contra el `f'c`/`fck` de diseño y mantener la granularidad que un ingeniero necesita para decidir. En planta, un modelo de este tipo sirve para **explorar dosificaciones** antes de invertir en ensayos destructivos costosos.


In [ ]:
TIPO_PROBLEMA = "regresion"
print(f"Tipo de problema elegido: {TIPO_PROBLEMA}")


## 2. Carga del dataset

El CSV ya viene convertido del XLS original de la UCI a columnas en español (ver `data/DATOS.md`), con 8 columnas de features de dosificación y 1 columna target (`Resistencia`).


In [ ]:
# Carga desde data/concrete.csv
RUTA_DATOS = Path("data/concrete.csv")
df = pd.read_csv(RUTA_DATOS)
print(f"Archivo: {RUTA_DATOS} | Forma: {df.shape[0]} filas × {df.shape[1]} columnas")


In [ ]:
N_FILAS_HEAD = 5
print(f"Primeras {N_FILAS_HEAD} mezclas:")
display(df.head(N_FILAS_HEAD))


## 3. Calidad de datos

El dataset no tiene valores nulos. Aun así, conviene revisar de cerca al menos una columna de dosificación: **Agua** es crítica porque la relación agua/cemento (W/C) es uno de los factores que más afecta la resistencia final, y un error de amasado ahí se traslada directo al resultado.


In [ ]:
COLUMNA_REVISAR = "Agua"
stats_col = df[COLUMNA_REVISAR].describe()
display(stats_col)


## 4. Estadísticas descriptivas

Mirando el resumen estadístico de las columnas clave, **Edad** es la que muestra mayor dispersión relativa (std frente a la media) — tiene sentido, porque el dataset mezcla probetas curadas entre 1 y 365 días. Una variable con tanta variabilidad exige, en general, más datos o mejores features para que el modelo la use bien.


In [ ]:
COLUMNAS_RESUMEN = ["Agua", "Edad", "Resistencia"]
resumen = df[COLUMNAS_RESUMEN].describe()
display(resumen)


## 5. Distribución del target (Resistencia)

El histograma de `Resistencia` muestra más mezclas por debajo de 40 MPa que por encima — el dataset incluye tanto probetas jóvenes (poco curado) como mezclas de baja resistencia, lo que genera un sesgo hacia la izquierda.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['Resistencia'], bins=30, color='#3498db', edgecolor='white')
ax.set_xlabel('Resistencia (MPa)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de resistencia a compresión')
plt.tight_layout()
plt.show()


In [ ]:
UMBRAL_RESISTENCIA = 40
n_fuertes = int((df['Resistencia'] >= UMBRAL_RESISTENCIA).sum())
n_debiles = int((df['Resistencia'] < UMBRAL_RESISTENCIA).sum())
print(f"≥ {UMBRAL_RESISTENCIA} MPa: {n_fuertes} | < {UMBRAL_RESISTENCIA} MPa: {n_debiles}")


## 6. Correlación entre variables

En la matriz de correlación, los ingredientes que más correlacionan (en valor absoluto) con `Resistencia` son **Cemento**, **Superplastificante** y **Edad**. El signo de **Agua** es negativo, lo cual es físicamente coherente: más agua diluye la mezcla y reduce la resistencia.


In [ ]:
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlación — dosificación vs resistencia')
plt.tight_layout()
plt.show()


In [ ]:
TOP_N_CORR = 3
corr_target = corr['Resistencia'].drop('Resistencia')
top_corr = corr_target.abs().sort_values(ascending=False).head(TOP_N_CORR).index.tolist()
for nombre in top_corr:
    print(f"  {nombre}: r = {corr_target[nombre]:.3f}")

signo = "negativa (coherente con la física del hormigón)" if corr_target['Agua'] < 0 else "positiva (inesperado, revisar datos)"
print(f"Agua: correlación {signo}.")


## 7. Relación física Agua vs Resistencia

El scatter de Agua vs Resistencia (coloreado por Edad) muestra la tendencia esperada: a mayor contenido de agua, menor resistencia. Esto ocurre porque el exceso de agua aumenta la porosidad y la relación agua/cemento, dos factores que debilitan la matriz de hormigón endurecido.


In [ ]:
def graficar_agua_resistencia(datos, titulo_extra=''):
    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(
        datos['Agua'], datos['Resistencia'],
        c=datos['Edad'], cmap='viridis', alpha=0.7, edgecolors='white', linewidth=0.3,
    )
    ax.set_xlabel('Agua (kg/m³)')
    ax.set_ylabel('Resistencia (MPa)')
    ax.set_title(f'Agua vs Resistencia {titulo_extra}')
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Edad (días)')
    plt.tight_layout()
    plt.show()


In [ ]:
EDAD_MIN_DIAS = 28
df_filtrado = df[df['Edad'] >= EDAD_MIN_DIAS]
n_filtradas = len(df_filtrado)
graficar_agua_resistencia(df_filtrado, f"(Edad ≥ {EDAD_MIN_DIAS} días, n={n_filtradas})")


## 8. Partición train / test

Como en el Lab 0: `X` son las features, `y` es `Resistencia`, y luego `model.fit(X_train, y_train)`. Fijamos `random_state=42` para que la partición sea reproducible entre corridas, y evaluamos sobre `X_test`/`y_test` — datos que el modelo nunca vio durante el entrenamiento — porque medir sobre los mismos datos de entrenamiento sobreestima el desempeño real.


In [ ]:
TARGET = "Resistencia"
FEATURES_TODAS = [c for c in df.columns if c != TARGET]
X = df[FEATURES_TODAS]
y = df[TARGET]
print(f"X: {X.shape} | y: {y.shape}")


In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")


## 9. Random Forest (hiperparámetros)

`n_estimators` controla cuántos árboles forman el bosque — más árboles suelen estabilizar la predicción a costa de más cómputo. Usamos las 8 features de dosificación, incluyendo **Edad**: si se quita esa columna, el R² del modelo cae de ~0.88 a ~0.39, porque el curado explica una parte enorme de la variabilidad en resistencia.


In [ ]:
N_ESTIMATORS = 100
MAX_DEPTH = 10
COLUMNAS_X = FEATURES_TODAS
X_train_sel = X_train[COLUMNAS_X]
X_test_sel = X_test[COLUMNAS_X]
modelo = RandomForestRegressor(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE)
modelo.fit(X_train_sel, y_train)
y_pred = modelo.predict(X_test_sel)
r2_test = r2_score(y_test, y_pred)
importancias = dict(zip(COLUMNAS_X, modelo.feature_importances_))
print(f"R² test = {r2_test:.3f}")


## 10. Feature importance y validación visual

Según el bosque aleatorio, **Edad** y **Cemento** suelen liderar la importancia de variables — el curado y la dosificación de ligante dominan el resultado. En obra, esto se traduce en priorizar esos factores al optimizar una dosificación: se pueden simular variantes (menos agua, más cemento o aditivo) antes de invertir en ensayos destructivos, siempre validando después con ensayos normativos reales.


In [ ]:
N_TOP_IMPORTANCIAS = 5  # Prueba 3 u 8

serie_imp = pd.Series(importancias).sort_values(ascending=False)
top_importancias = serie_imp.head(N_TOP_IMPORTANCIAS)
importancias_ordenadas = top_importancias.index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_importancias.iloc[::-1].plot(kind='barh', ax=axes[0], color='#2ecc71')
axes[0].set_title(f'Top {N_TOP_IMPORTANCIAS} — Feature Importance')
axes[0].set_xlabel('Importancia relativa')

axes[1].scatter(y_test, y_pred, alpha=0.6, edgecolors='white')
lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[1].plot(lim, lim, 'r--', label='Predicción perfecta')
axes[1].set_xlabel('Resistencia real (MPa)')
axes[1].set_ylabel('Resistencia predicha (MPa)')
axes[1].set_title(f'Validación visual (R² = {r2_test:.3f})')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Top importancias:", importancias_ordenadas)


## Reflexión: preguntas que los alumnos necesitarían

- ¿Qué pasaría si este modelo se usa para predecir la resistencia de una mezcla con ingredientes fuera del rango de dosificación visto en entrenamiento (por ejemplo, un aditivo nuevo no presente en el dataset UCI)?
- La importancia de variables dice *qué tanto* usa el modelo cada ingrediente para predecir, pero no explica *por qué* químicamente (hidratación del cemento, relación agua/cemento, etc.). ¿Cómo complementarías este modelo con conocimiento de ingeniería de materiales?
- Antes de confiar en una predicción del modelo para una decisión real (por ejemplo, aprobar una dosificación), ¿qué ensayo de laboratorio usarías para contrastarla, y con qué tolerancia de error?
- El dataset tiene 1030 mezclas de un solo estudio. ¿Qué riesgos hay al aplicar este modelo a hormigones de otra región, otro cemento o otro clima de curado?
- Si tuvieras que reducir el costo de una mezcla sin perder resistencia, ¿cómo usarías el gráfico de importancia de variables para decidir qué ingrediente ajustar primero?
